In [ ]:
from datetime import datetime, timezone

def timestamp(time_stmp):
    """Parse a JSON timestamp and return a timezone-aware UTC datetime."""
    if not time_stmp:
        return None

    # Support ISO timestamps ending in Z as well as the current
    # format: 2025-12-21 09:19:00.
    dt = datetime.fromisoformat(str(time_stmp).replace("Z", "+00:00"))

    # The source JSON has no timezone, so interpret it as UTC.
    if dt.tzinfo is None:
        dt = dt.replace(tzinfo=timezone.utc)

    return dt.astimezone(timezone.utc)

In [ ]:
import json
from pathlib import Path
from langchain_community.document_loaders import JSONLoader

file_path = Path(r"C:\Coding\Projects\RAG Project\sample_json_20260301\20251221.json")

def add_timestamp_metadata(record, metadata):
    parsed_time = timestamp(record.get("timestamp"))
    # Store metadata as a string so vector databases can serialize it.
    metadata["timestamp"] = parsed_time.isoformat() if parsed_time else None
    return metadata

loader = JSONLoader(
    file_path=str(file_path),
    jq_schema=".",
    text_content=False,
    json_lines=True,
    metadata_func=add_timestamp_metadata,
)

documents = loader.load()

print(f"Loaded {len(documents):,} log records")
for document in documents[:3]:
    print(document.metadata["timestamp"])


In [ ]:
import json
import os
import time
from pathlib import Path
from uuid import NAMESPACE_URL, uuid5
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient, models
from qdrant_client.http.exceptions import ResponseHandlingException

load_dotenv(r"C:\Coding\Projects\RAG Project\processing\.env")
QDRANT_URL = os.getenv("QDRANT_URL")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY") or None
COLLECTION_NAME = os.getenv("QDRANT_COLLECTION", "sim001_logs_bge_base_en")
MODEL_NAME = "BAAI/bge-base-en"
BATCH_SIZE = 256
MAX_RETRIES = 4
CHECKPOINT_PATH = Path(r"C:\Coding\Projects\RAG Project\processing\.qdrant_ingest_checkpoint.json")

log_documents = documents
embeddings = HuggingFaceEmbeddings(model_name=MODEL_NAME, model_kwargs={"device": "cuda"})
client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY, timeout=120)

collection_created = not client.collection_exists(COLLECTION_NAME)
if collection_created:
    vector_size = len(embeddings.embed_query("log event"))
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=models.VectorParams(size=vector_size, distance=models.Distance.COSINE),
    )

vector_store = QdrantVectorStore(
    client=client, collection_name=COLLECTION_NAME, embedding=embeddings,
)
if "metadata.timestamp" not in client.get_collection(COLLECTION_NAME).payload_schema:
    client.create_payload_index(
        collection_name=COLLECTION_NAME,
        field_name="metadata.timestamp",
        field_schema=models.PayloadSchemaType.DATETIME,
    )

run_key = str(uuid5(
    NAMESPACE_URL,
    f"{QDRANT_URL}|{COLLECTION_NAME}|{MODEL_NAME}|{file_path.resolve()}|"
    f"{file_path.stat().st_size}|{len(log_documents)}",
))
checkpoint = json.loads(CHECKPOINT_PATH.read_text(encoding="utf-8")) if CHECKPOINT_PATH.exists() else {}
start_at = checkpoint.get("next_start", 0) if checkpoint.get("run_key") == run_key else 0
if collection_created or (start_at and client.count(COLLECTION_NAME, exact=True).count < start_at):
    start_at = 0
print(f"Resuming at record {start_at:,} of {len(log_documents):,}")

for start in range(start_at, len(log_documents), BATCH_SIZE):
    batch = log_documents[start:start + BATCH_SIZE]
    ids = [
        str(uuid5(NAMESPACE_URL, f"{doc.metadata['source']}:{doc.metadata['seq_num']}"))
        for doc in batch
    ]
    for attempt in range(MAX_RETRIES):
        try:
            vector_store.add_documents(documents=batch, ids=ids, batch_size=BATCH_SIZE)
            break
        except ResponseHandlingException as exc:
            if "timed out" not in str(exc).lower() or attempt == MAX_RETRIES - 1:
                raise
            delay = min(2 ** attempt, 8)
            print(f"Upload timed out at record {start + 1:,}; retrying in {delay}s")
            time.sleep(delay)

    end = start + len(batch)
    temp_path = CHECKPOINT_PATH.with_suffix(".tmp")
    temp_path.write_text(json.dumps({"run_key": run_key, "next_start": end}), encoding="utf-8")
    temp_path.replace(CHECKPOINT_PATH)
    if end - start_at <= BATCH_SIZE * 10 or (end - start_at) % (BATCH_SIZE * 10) == 0 or end == len(log_documents):
        print(f"Stored {end:,} / {len(log_documents):,} logs in {COLLECTION_NAME}")

results = vector_store.similarity_search("failed login and unusual network activity", k=3)
for doc in results:
    print(doc.metadata.get("timestamp"), doc.page_content[:250])

In [ ]:
retriever = vector_store.as_retriever(search_kwargs={"k": 5})

matches = retriever.invoke("At 3:00 AM security faliure")
for doc in matches:
    print(doc.metadata.get("timestamp"), doc.page_content[:300])


In [ ]:
import sys
import os
from pathlib import Path
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_qdrant import QdrantVectorStore

sys.path.insert(0, str(Path(r"C:\Coding\Projects\RAG Project\processing")))
from query_pipeline import (
    understand_question, possible_incidents, retrieve_and_rank, generate_forensic_answer,
)

load_dotenv(r"C:\Coding\Projects\RAG Project\processing\.env")
groq_key = os.getenv("GROQ_API_KEY") or os.getenv("GRPQ_API_KEY")
if not groq_key:
    raise RuntimeError("Set GROQ_API_KEY in processing/.env")
llm = ChatGroq(model="qwen/qwen3.8-27b", temperature=0, groq_api_key=groq_key)

# Reconnect for search only. This does not rerun the ingestion cell.
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-base-en", model_kwargs={"device": "cuda"}
)
vector_store = QdrantVectorStore.from_existing_collection(
    collection_name=os.getenv("QDRANT_COLLECTION", "sim001_logs_bge_base_en"),
    embedding=embeddings,
    url=os.environ["QDRANT_URL"],
    api_key=os.getenv("QDRANT_API_KEY"),
    timeout=120,
)
print("Connected to Groq and the existing Qdrant collection (no upload).")


In [ ]:
# Ask a question after running the connection cell above.
# Use the user's actual timezone for relative dates such as 'last Thursday'.
question = input("Ask a log-investigation question: " )
plan = understand_question(question, llm, user_timezone="Asia/Kolkata")
print("Parsed plan:", plan.model_dump())

if plan.incident_description:
    print("Possible incident logs (inspect before confirming a timestamp):")
    incident_hits = possible_incidents(plan, vector_store)
    if not incident_hits:
        print("No incident candidates in the indexed logs.")
    for doc, score in incident_hits:
        print(doc.metadata.get("timestamp"), doc.metadata.get("seq_num"), doc.page_content[:250])
    print("No 'before' search runs until you confirm an actual incident log.")
else:
    ranked = retrieve_and_rank(plan, vector_store)
    if not ranked:
        print("No matching logs in the indexed subset. Check the date and time zone.")
    for score, doc in ranked[:5]:
        print(doc.metadata.get("timestamp"), doc.metadata.get("seq_num"), round(score, 4))
        print(doc.page_content[:300], "\n")
    if ranked:
        approval = input("Send up to 8 retrieved log texts (may include IPs/accounts) to Groq? Type YES: " )
        if approval == "YES":
            print("Forensic answer:\n", generate_forensic_answer(
                question, ranked, llm, allow_external_log_upload=True
            ))
        else:
            print("No log text was sent to Groq.")


In [ ]:
# Run this only when the candidate list above includes the actual incident.
# Do not select an unrelated log just because it has a high similarity score.
from datetime import datetime

if not plan.incident_description:
    print("This question already used an explicit time window.")
else:
    selected_seq_num = int(input("Confirmed incident log seq_num: "))
    confirmed = next(
        (doc for doc, _ in incident_hits if doc.metadata.get("seq_num") == selected_seq_num),
        None,
    )
    if confirmed is None:
        raise ValueError("Choose the seq_num of a shown incident candidate")
    incident_time = datetime.fromisoformat(confirmed.metadata["timestamp"])
    ranked = retrieve_and_rank(
        plan, vector_store, confirmed_incident_time=incident_time
    )
    for score, doc in ranked[:5]:
        print(doc.metadata.get("timestamp"), doc.metadata.get("seq_num"), round(score, 4))
        print(doc.page_content[:300], "\n")
    if ranked:
        approval = input("Send the incident and up to 8 retrieved logs (may include IPs/accounts) to Groq? Type YES: " )
        if approval == "YES":
            print("Forensic answer:\n", generate_forensic_answer(
                question, ranked, llm, incident_doc=confirmed,
                allow_external_log_upload=True,
            ))
        else:
            print("No log text was sent to Groq.")
